# Four bugs no test could catch

*Question, Intuition, Math, Code, Assumptions, How it breaks*

This project runs 236 tests, `mypy --strict`, `ruff`, and a pandera schema on
every single read and every single write.

It has still shipped four defects that changed the answer.

Not one of them raised. Not one of them failed a test. Every one produced output
that looked completely reasonable, which is why they survived.

Section 4 checks whether the data you are about to read in every other chapter is
carrying more of them **right now**. That check gets computed on the spot rather
than written down, so this page cannot go stale, and it cannot flatter itself.

## 1. Question

If the type checker passes, the linter passes, the schemas validate and all 236
tests are green, what class of error is still getting through? And what would
actually catch it?

In [1]:
import warnings

import matplotlib
import pandas as pd

from gambeta import laws

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")

print(f"{len(seasons):,} player-seasons")
print(f"{len(ranking):,} players ranked")

39,877 player-seasons
5,508 players ranked


## 2. Intuition

There are two ways a number can be wrong, and only one of them is a programming
error.

**Syntactically wrong.** A string where a float belongs, a missing column, a
negative count, a duplicated primary key. Machines find these easily, because "is
this a float" is a question about the code. Types, schemas and tests are all
built for exactly this, and on this project they genuinely work. The pandera
layer has caught real breakage at the seam between stages.

**Semantically wrong.** The code computes exactly what it says, and what it says
is the wrong quantity. `consistency = -std(scores)` is a perfectly correct
standard deviation. It is also a measure that rewards being mediocre every
season. No type system has an opinion about that, because nothing is *broken*.

The trap underneath is worse than it first looks. **A test asserts what you
believed when you wrote it.** If the belief was wrong, the test is wrong in
exactly the same direction, passes forever, and now stacks false confidence on
top of the original error.

Let me make that concrete. The frame below is the shape that fabricated a player
with nineteen thousand minutes in a thirty-eight match season, and the project's
own schema is perfectly happy with it.

In [2]:
def michel(n: int = 2) -> pd.DataFrame:
    """`n` different people, same name, same club, same season.

    Two players called Míchel really were at Rayo Vallecano in 2002-03. Only
    their birth year separates them.
    """
    base = dict(
        league="ESP-La Liga",
        season="0203",
        team="Rayo Vallecano",
        player="Michel",
        nation="es",
        pos="MF",
        age=26.0,
        mp=30,
        starts=25,
        minutes=2200,
        goals=3,
        assists=2,
        npg=3,
        pk=0,
        pkatt=0,
        yellow=5,
        red=0,
        sot=10.0,
        sot_p90=0.4,
        g_per_sot=0.3,
        min_pct=64.0,
        complete=20.0,
        subs=5.0,
        second_yellow=0.0,
        fouls=30.0,
    )
    return pd.DataFrame([{**base, "born": 1975.0 + i} for i in range(n)])


laws.OUTFIELD_RAW.validate(michel())
print("OUTFIELD_RAW accepts it. Correctly, because these are two different people.")
print("\nThe rows are fine. What happens to them next is not.")

OUTFIELD_RAW accepts it. Correctly, because these are two different people.

The rows are fine. What happens to them next is not.


## 3. The four bugs

Each one is stated as *what it computed* against *what it was supposed to mean*.
That gap is the whole problem. In three of the four cases the computation is not
merely correct, it is the obvious implementation of the name.

In [3]:
pd.DataFrame(
    [
        {
            "bug": "consistency",
            "computed": "-std(season scores)",
            "meant": "did he have bad years?",
            "caught by": "reading the output",
        },
        {
            "bug": "reliability",
            "computed": "completed matches / starts",
            "meant": "is he picked to start?",
            "caught by": "reading the output",
        },
        {
            "bug": "age",
            "computed": "attached by row position",
            "meant": "attached by player",
            "caught by": "an invariant",
        },
        {
            "bug": "award winners",
            "computed": "all Ballon d'Or winners",
            "meant": "the men's award only",
            "caught by": "an external check",
        },
    ]
).set_index("bug")

,computed,meant,caught by
bug,,,
consistency,-std(season scores),did he have bad years?,reading the output
reliability,completed matches / starts,is he picked to start?,reading the output
age,attached by row position,attached by player,an invariant
award winners,all Ballon d'Or winners,the men's award only,an external check


Two of the four were catchable by a rule a machine could check. Two were not
catchable by any rule, and got found by a person looking at a list of names and
thinking *that cannot possibly be right.*

`consistency` disqualified Messi, Ronaldo, Kane, Haaland, Lewandowski, Suárez,
Henry and Salah in a single run, and promoted the most featureless players in the
dataset. I work through it properly in the requirements-and-gates chapter.

The rest of this one is about the other half: the bugs a machine *could* have
caught, and why it did not.

## 4. Code

### The mechanism

`join_side_tables` merges three secondary FBref tables onto the standard one, on
`(league, season, team, player)`. That key identifies a row uniquely, right up
until two people share a name at the same club.

Here is the thing I had not internalised: a left join is not a lookup. When both
sides hold duplicate keys, pandas returns the **cross product**, and the row count
multiplies at every join.

In [4]:
from gambeta.scouts.fbref import _INDEX

standard = michel()[[*_INDEX, "born", "minutes", "mp"]]
sides = {
    "shooting": michel()[[*_INDEX, "sot"]],
    "playing_time": michel()[[*_INDEX, "complete"]],
    "misc": michel()[[*_INDEX, "fouls"]],
}

out = standard
print(f"  standard{'':<22}{len(out):>4} rows")
for name, frame in sides.items():
    out = out.merge(frame, on=_INDEX, how="left")
    print(f"  after joining {name:<16}{len(out):>4} rows")

print(f"\n2 people -> {len(out)} rows. Two to the power of the number of joins.")

  standard                         2 rows
  after joining shooting           4 rows
  after joining playing_time       8 rows
  after joining misc              16 rows

2 people -> 16 rows. Two to the power of the number of joins.


Nothing raises. Every row is schema-valid, every column has the right dtype, no
count is negative. The frame is just **fourteen rows longer than the truth**, and
those rows carry real numbers that the next stage will faithfully add up.

In [5]:
collapsed = out["minutes"].sum()
print(f"true minutes for the two players: {standard['minutes'].sum():>7,}")
print(f"minutes after collapsing         : {collapsed:>7,}")
print(f"\nInflation factor: {collapsed / standard['minutes'].sum():.0f}x")
print("\nThat figure becomes the row's weight in the minutes-weighted career")
print("average, and the denominator of every per-90 rate computed from it.")

true minutes for the two players:   4,400
minutes after collapsing         :  35,200

Inflation factor: 8x

That figure becomes the row's weight in the minutes-weighted career
average, and the denominator of every per-90 rate computed from it.


### The fix, and why it is two characters of judgement

pandas already knows how to refuse this. `validate="one_to_one"` asserts that the
join is a lookup rather than a product, and turns a silent multiplication into an
exception.

In [6]:
try:
    standard.merge(sides["shooting"], on=_INDEX, how="left", validate="one_to_one")
except pd.errors.MergeError as exc:
    print(f"MergeError: {exc}")

KEY = [*_INDEX, "born"]
fixed = standard
for frame in (michel()[[*KEY, "sot"]], michel()[[*KEY, "complete"]], michel()[[*KEY, "fouls"]]):
    fixed = fixed.merge(frame, on=KEY, how="left", validate="one_to_one")

print(f"\nwith `born` in the key: {len(fixed)} rows, {fixed['minutes'].sum():,} minutes. Correct.")

MergeError: Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in left:
      league season           team player
ESP-La Liga   0203 Rayo Vallecano Michel ...
Duplicates in right:
      league season           team player
ESP-La Liga   0203 Rayo Vallecano Michel ...

with `born` in the key: 2 rows, 4,400 minutes. Correct.


### What the published data says about itself

The checks below run against the committed sample, which is the same file every
other chapter in this book reads. They are not illustrations. Whatever they print
is the state of this repository at the moment you ran it, and nothing in this
chapter tells you in advance what that will be. **The verdict is computed, not
written.**

These are **logical** invariants. Not judgements about football, not thresholds
somebody picked. Each one is a statement that cannot be false in any possible
world: you cannot complete more matches than you played, and a share of your
team's minutes cannot exceed all of them.

In [7]:
def holds(condition: pd.Series, *inputs: pd.Series) -> pd.Series:
    """A check is violated only where its inputs are known.

    Missing is not a violation. Conflating the two turns "FBref never recorded
    this" into "the data is broken", which is the same mistake as turning it
    into a zero, just in the other direction.
    """
    known = pd.concat(inputs, axis=1).notna().all(axis=1)
    return condition | ~known


LOGICAL = {
    "a share of team minutes never exceeds 1": holds(
        seasons["availability"] <= 1.0, seasons["availability"]
    ),
    "cannot start more matches than played": holds(
        seasons["starts"] <= seasons["mp"], seasons["starts"], seasons["mp"]
    ),
    "cannot complete more matches than played": holds(
        seasons["complete"] <= seasons["mp"], seasons["complete"], seasons["mp"]
    ),
    "one row per player-season": ~seasons.duplicated(["player_id", "season"]),
}

# Declared somewhere the pipeline enforces it, rather than only believed.
DECLARED = {"one row per player-season"}  # laws.PLAYER_SEASON, unique=[player_id, season]

broken = {label: int((~holds).sum()) for label, holds in LOGICAL.items() if not holds.all()}

for label, holds_ in LOGICAL.items():
    mark = "declared" if label in DECLARED else "believed"
    bad = int((~holds_).sum())
    print(f"  {'PASS' if bad == 0 else 'FAIL'}  [{mark}]  {label:<44}{bad:>4} violations")

print()
if broken:
    print(f"{len(broken)} of {len(LOGICAL)} logical invariants are violated by the published data.")
    stated = len(broken.keys() & DECLARED)
    print(f"Of the {len(broken)} violated, {stated} were declared in the codebase.")
    print(f"largest share of a team's minutes recorded: {seasons['availability'].max():.1%}")
else:
    print(
        f"All {len(LOGICAL)} hold. Every one of them also holds in `laws.py`,"
        " which is why they hold here."
    )

  PASS  [believed]  a share of team minutes never exceeds 1        0 violations
  PASS  [believed]  cannot start more matches than played          0 violations
  FAIL  [believed]  cannot complete more matches than played      11 violations
  PASS  [declared]  one row per player-season                      0 violations

1 of 4 logical invariants are violated by the published data.
Of the 1 violated, 0 were declared in the codebase.
largest share of a team's minutes recorded: 100.0%


Read the `declared` / `believed` column, because that is the argument in
miniature.

One of those four is **declared**. `PLAYER_SEASON` carries
`unique=["player_id", "season"]`, so the pipeline physically cannot emit a
duplicate without stopping. The other three are true statements about arithmetic
that nothing in this project asserts anywhere.

**Whether an undeclared invariant holds is luck.** The cell above tells you
today's luck, and it will keep telling you. Including about bugs that do not
exist yet, and including on the day I would rather it did not.

### The invariant this chapter got wrong

The first version of that cell asserted something else: **completed matches
cannot exceed starts.** It reads like arithmetic. It is not. It is a claim about
football, and it is false.

A substitute who comes on and is still on the pitch at the final whistle has
completed a match he did not start. FBref counts him. The bound is appearances,
not starts, and the difference shows up in the data.

In [8]:
known = seasons[seasons["complete"].notna()]
past_starts = known["complete"] > known["starts"]
past_played = known["complete"] > known["mp"]

print(f"complete > starts : {int(past_starts.sum()):>3}   <- the wrong bound")
print(f"complete > mp     : {int(past_played.sum()):>3}   <- the real bound, and still broken")
legit = int((past_starts & ~past_played).sum())
print(f"difference        : {legit:>3}   <- legitimate football\n")
known.loc[past_starts & ~past_played, ["player", "season", "mp", "starts", "complete"]].head(5)

complete > starts :  20   <- the wrong bound
complete > mp     :  11   <- the real bound, and still broken
difference        :   9   <- legitimate football



,player,season,mp,starts,complete
8517,Ángel Morales,0001,19.0,7.0,10.0
9584,Neru,0304,20.0,19.0,20.0
11573,Henrique,0910,22.0,20.0,21.0
16969,Massimiliano Esposito,0001,18.0,7.0,8.0
17521,Cristian Stellini,0203,29.0,28.0,29.0


Those are substitutes who finished matches. Ángel Morales played nineteen,
started seven, and was on the pitch at the end of ten, so three of them without
starting. Under my invariant he was a data error. He is a squad player who got
minutes.

What survives the correction is smaller and genuinely impossible: rows where a
player completed more matches than he **appeared in**. Those are FBref
disagreeing with itself, and no join of mine produced them.

Notice what happened there. A wrong belief, written down as an invariant, does
not sit quietly. It accuses correct data of being broken. That is still better
than the alternative, because it is visible and you can argue with it. But it is
the same class of error as the four bugs above, committed by the person writing
the chapter about them, inside the chapter itself.

### Logical invariants are free. Domain invariants are a guess.

The obvious next check is a bound: nobody plays more than 38 league matches in a
season. It looks like the same kind of rule. It is not.

In [9]:
cap = seasons["mp"] <= 38
multi_club = seasons["teams"].str.contains(",")

print(f"rows failing 'at most 38 appearances' : {int((~cap).sum()):>4}")
print(f"  ...that span more than one club     : {int((~cap & multi_club).sum()):>4}   <- ambiguous")
alone = int((~cap & ~multi_club).sum())
print(f"  ...at a single club                 : {alone:>4}   <- impossible")
print(f"\n{int(multi_club.sum()):,} rows collapse a mid-season transfer, and their")
print("appearances are summed across clubs. For those, 38 is not the ceiling.")

rows failing 'at most 38 appearances' :    8
  ...that span more than one club     :    8   <- ambiguous
  ...at a single club                 :    0   <- impossible

1,892 rows collapse a mid-season transfer, and their
appearances are summed across clubs. For those, 38 is not the ceiling.


The count on its own cannot separate the two cases. Appearances get summed
across clubs for a transfer season, so some of those violations are legitimate
football and some are not, and a bound has no way of telling you which. It needs
a carve-out, and the carve-out needs a judgement about how many leagues a player
can appear in inside one season.

The key check needs none of that. `validate="one_to_one"` is a fact about
joining, not a fact about football, and it catches every fabricated row exactly.

**Prefer invariants about structure over invariants about values.** Structural
ones are free, exact, and there is nothing to argue about.

## 5. Assumptions

1. **The invariant encodes a true belief.** This is the entire load-bearing
   assumption, and section 6 is about what happens when it does not.
2. **Violations are visible.** A check that runs in a notebook nobody opens is
   not a check. These belong in the schema, where the pipeline cannot get past
   them.
3. **Structure is knowable.** "One row per player-season" is checkable because I
   decided the grain deliberately. On a dataset whose grain nobody wrote down,
   there is nothing to assert.

## 6. How it breaks

An invariant is a belief with an exception attached. A wrong belief does not
become right by being enforced. It becomes **load-bearing**, and now it deletes
data loudly instead of quietly.

Here is one that any reasonable person might write. Composite scores are
standardised, so they ought to behave like a distribution. Five standard
deviations is already extraordinary. Reject anything beyond it as a data error.

In [10]:
scores = ranking["score"].to_numpy(dtype=float)
sigma = (scores - scores.mean()) / scores.std(ddof=0)

rejected = ranking.loc[sigma > 5.0, ["player", "score"]].copy()
rejected["sigma"] = sigma[sigma > 5.0]

print("'no player exceeds 5 sigma' would reject:\n")
print(rejected.round(2).to_string(index=False))
print("\n(Nothing above is applied anywhere. This is what the rule *would* do.)")

'no player exceeds 5 sigma' would reject:

           player  score  sigma
     Lionel Messi   3.33   5.89
Cristiano Ronaldo   2.93   5.18

(Nothing above is applied anywhere. This is what the rule *would* do.)


The rule is statistically literate, easy to defend in a code review, and it
throws away the answer to the question this whole project exists to ask.

It fails because it smuggles in a distributional assumption the data does not
satisfy. Football ability is heavy-tailed. The distribution chapter shows a
normal model implying the best player should occur once in six hundred million,
out of a pool of five and a half thousand. Under that model Messi is not a
footballer, he is an outlier to be cleaned.

Here is the distinction that survives all of this:

| | Asserts | Can it be wrong? |
|---|---|---|
| `starts <= mp` | arithmetic | no |
| `unique(league, season, team, player, born)` | the grain of the table | only if the grain is misdocumented |
| `complete <= starts` | arithmetic, apparently. Football, actually | yes, and it was |
| `mp <= 38` | a fact about competitions | yes, because of transfers |
| `score <= 5 sigma` | a distribution | yes, and catastrophically |

The third row is the dangerous one, because it is the hardest to tell apart from
the first. Both look like counting. Only one of them is.

### The lesson

The four bugs shared one property: **plausible output, no exception, every test
green.** No amount of additional testing would have found them, because the tests
were written by the same person holding the same wrong belief.

Two defences work, and they are different in kind.

**Structural invariants**, for anything true by arithmetic or by the declared
grain of a table. Free, exact, and they belong in `laws.py` rather than in a
notebook.

**Looking at the answer**, for everything else. This is why the failure table,
`unresolved.csv` and the awards check are published outputs rather than internal
diagnostics. They exist so that a wrong answer has somewhere to become visible.
On this project, that has been the only defence that ever actually worked.